# 01 · Load and clean the LendingClub loans

**What this notebook does, in plain words**

1. Finds the big LendingClub file in the `data` folder.
2. Loads only the columns we need (the full file has 151 columns).
3. Keeps only loans whose ending we know: 36-month loans issued 2007–2015 that were either paid off or written off.
4. Turns messy text columns (like `"10+ years"`) into numbers.
5. Saves a small, clean file that the next notebooks load in seconds.

Run the cells top to bottom with **Shift+Enter**, or click **Run All**.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data")

# Kaggle's zip can have nested folders, so search the whole data folder for the file.
candidates = [p for p in DATA_DIR.rglob("accepted_2007_to_2018*") if p.is_file()]
# Prefer the plain .csv (faster to read) over the compressed .gz
candidates.sort(key=lambda p: (p.suffix == ".gz", str(p)))

if not candidates:
    raise FileNotFoundError(
        "Couldn't find accepted_2007_to_2018Q4.csv (or .csv.gz) inside the data folder. "
        "Download the 'Lending Club' dataset by wordsforthewise from Kaggle and unzip it into data/."
    )

DATA_FILE = candidates[0]
print("Using:", DATA_FILE)

## Which columns we load, and why

We split the columns into four groups. **Only the first group is allowed into our model.**

- **Application:** what LendingClub knew on the day the loan was made.
- **Platform's answer:** LendingClub's own grade and interest rate. This is the baseline we compete against, so the model never sees it.
- **What happened later:** whether the loan was repaid and how much came back. Using these to predict would be cheating (this is called *leakage*). We keep them only to score results.
- **Job title:** saved for the stereotype test, kept out of the main model.

In [ ]:
APPLICATION = [
    "loan_amnt", "fico_range_low", "fico_range_high", "annual_inc", "dti",
    "revol_bal", "revol_util", "open_acc", "total_acc", "delinq_2yrs",
    "inq_last_6mths", "pub_rec", "emp_length", "home_ownership", "purpose",
    "earliest_cr_line",
]
PLATFORMS_ANSWER = ["grade", "sub_grade", "int_rate"]
WHAT_HAPPENED_LATER = ["loan_status", "funded_amnt", "total_pymnt"]
JOB = ["emp_title"]
IDS_AND_DATES = ["id", "issue_d", "term"]

usecols = IDS_AND_DATES + APPLICATION + PLATFORMS_ANSWER + WHAT_HAPPENED_LATER + JOB

raw = pd.read_csv(DATA_FILE, usecols=usecols, low_memory=False)
print(f"{len(raw):,} rows loaded")

## Keep only loans whose ending we know

- **36-month loans issued 2007–2015:** the data stops at the end of 2018, so these loans had all reached their end date.
- **Status "Fully Paid" or "Charged Off":** every loan we keep has a known outcome. *Charged Off* means LendingClub gave up collecting, which is what we call a default.
- We drop the small group marked *"Does not meet the credit policy"*, which came from an older approval policy.
- The file ends with a few summary lines that aren't loans; they have no valid date, so they get dropped automatically.

In [ ]:
df = raw.copy()

df["issue_date"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
df = df.dropna(subset=["issue_date"])          # removes the summary lines at the bottom of the file
df["year"] = df["issue_date"].dt.year
df["term_months"] = pd.to_numeric(df["term"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")

keep = (
    (df["term_months"] == 36)
    & df["year"].between(2007, 2015)
    & df["loan_status"].isin(["Fully Paid", "Charged Off"])
)
df = df.loc[keep].copy()
df["default"] = (df["loan_status"] == "Charged Off").astype(int)

print(f"{len(df):,} finished 36-month loans kept")

## Turn messy text into numbers

| Column | Looks like | Becomes |
| --- | --- | --- |
| `emp_length` | `"< 1 year"`, `"10+ years"` | `emp_years`: 0 to 10 |
| `earliest_cr_line` | `"Aug-2003"` | `credit_history_years`: years of credit history on the day of the loan |
| `fico_range_low` / `high` | two numbers | `fico`: their middle |
| `int_rate`, `revol_util` | sometimes `"13.5%"` | plain numbers |
| `emp_title` | `"  Registered Nurse "` | `job_title`: `"registered nurse"` |

In [ ]:
def to_number(series):
    # Strip a % sign if there is one, then convert; anything unreadable becomes NaN (missing)
    return pd.to_numeric(series.astype(str).str.replace("%", "", regex=False).str.strip(), errors="coerce")

for col in ["int_rate", "revol_util"]:
    df[col] = to_number(df[col])

# "< 1 year" -> 0, "10+ years" -> 10, missing stays missing
df["emp_years"] = pd.to_numeric(
    df["emp_length"].astype(str).str.replace("< 1", "0", regex=False).str.extract(r"(\d+)")[0],
    errors="coerce",
)

first_credit = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y", errors="coerce")
df["credit_history_years"] = (df["issue_date"] - first_credit).dt.days / 365.25

df["fico"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

df["job_title"] = df["emp_title"].astype(str).str.lower().str.strip()
df.loc[df["job_title"].isin(["nan", ""]), "job_title"] = np.nan

df[["emp_length", "emp_years", "earliest_cr_line", "credit_history_years", "fico", "job_title"]].head()

## Sanity checks

If the data loaded correctly, the default rate should climb steadily from grade A to grade G. If it doesn't, something went wrong above.

In [ ]:
by_year = df.groupby("year").agg(loans=("default", "size"), default_rate_pct=("default", "mean"))
by_year["default_rate_pct"] = (by_year["default_rate_pct"] * 100).round(1)
by_year

In [ ]:
by_grade = df.groupby("grade").agg(loans=("default", "size"), default_rate_pct=("default", "mean"))
by_grade["default_rate_pct"] = (by_grade["default_rate_pct"] * 100).round(1)
by_grade

## Save the clean file

The next notebook loads this in seconds instead of minutes. It goes in `data/`, which `.gitignore` keeps off GitHub.

In [ ]:
KEEP = (
    ["id", "issue_date", "year", "default", "loan_status"]
    + ["loan_amnt", "fico", "annual_inc", "dti", "revol_bal", "revol_util", "open_acc", "total_acc",
       "delinq_2yrs", "inq_last_6mths", "pub_rec", "emp_years", "credit_history_years",
       "home_ownership", "purpose"]
    + ["grade", "sub_grade", "int_rate", "funded_amnt", "total_pymnt", "job_title"]
)
df[KEEP].to_parquet(DATA_DIR / "loans_clean.parquet", index=False)
print(f"Saved {len(df):,} loans to data/loans_clean.parquet")